# Reduce windowed TCC metrics to regions

Converts cached 2-D window diagnostics into area-weighted regional ensemble statistics.


In [ ]:
from pathlib import Path
INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_out")

groups = ("Jan2012", "Jun2012")
run_by_group = {"Dec2011": ("da",), "Jan2012": ("fc",), "Jun2012": ("fc",)}
selected = {"ERA5": ("TREFHT",), "GPM": ("PRECT",)}
force_compute = False
member_quantiles = (0.05, 0.25, 0.50, 0.75, 0.90, 0.95)
verbose = True


window_scheme = "biweekly"  # "biweekly", "weekly", "15day", or "month"


In [ ]:
from pathlib import Path
import sys

repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))

from configs.output_paths import workflow_output_dirs
from configs.s2s_config import EXPERIMENT_GROUPS, OBS_VARIABLES, PCC_REGIONS, TCC_REGIONS, LEAD_WINDOWS, WEEKLY_WINDOWS, FIFTEEN_DAY_WINDOWS, MONTH_WINDOWS, WINDOW_SCHEMES, AVAILABLE_PCC_METRICS, AVAILABLE_PCC_MAP_METRICS, AVAILABLE_TCC_METRICS
from util.s2s_experiments import build_experiments
from util.s2s_observations import OBS_REGISTRY, get_obs_file


## Notes: available selections

- Experiment groups: Dec2011 (analysis), Jan2012, and Jun2012 (forecasts).
- Runs: da for Dec2011; fc (standard forecast) or wc (water-cycle forecast) for forecast groups.
- Observations and variables: `ERA5` supports UBOT, VBOT, PRECT, PSL, T925, Q925, U850, V850, T850, Q850, Z500, OMEGA850/500/200, U/V500, U/V200, FLDS/C, FLNS/C, FSDS/C, FSNS/C, LHFLX, SHFLX, QFLX, TREFHT, TS, PRECSL, PRECC, and PRECL; `GPCP` and `GPM` support PRECT; `NOAA-OLR` supports FLUT.
- Regions: GLOBAL, land/ocean where applicable, NINO3.4, MAR_CONT, CONUS, NH_POLAR.
- Existing NetCDF diagnostics are reused. Set `force_compute = True` to rebuild selected products.

Available reductions include ensemble-mean TCC/RMSE/bias and configurable pre/post-spatial member quantiles.

Exact base-map fields are `TCC`, `RMSE`, and `BIAS` for members and `TCC_ensmean`, `RMSE_ensmean`, and `BIAS_ensmean` for the ensemble mean. For each base metric, regional outputs provide `<METRIC>_regional_member`, `<METRIC>_member_ens_mean`, `<METRIC>_member_ens_std`, `<METRIC>_member_qpostXX`, `<METRIC>_member_qpreXX`, `<METRIC>_regional_ensmean`, valid-weight and valid-fraction fields, plus `region_area_weight_sum`. `XX` is each configured quantile percentile.


In [ ]:
from util.s2s_tcc_metrics import RegionSpec, RegionalReduceConfig, S2SRegionalSkillReducer

base_dir, figure_dir = workflow_output_dirs("fcst_s2s_tcc", f"{window_scheme}_metrics", output_root=DIAGNOSTIC_OUTPUT_ROOT)
regional_dir = base_dir / "regional"; regional_dir.mkdir(parents=True, exist_ok=True)
land_mask = INPUT_DATA_ROOT / "reference/lnd_sea_mask/landmask_1x1.nc"
regions = tuple(RegionSpec(name=r["name"], lat=r["lat"], lon=r["lon"], mask=r.get("mask","none")) for r in TCC_REGIONS)
reducer = S2SRegionalSkillReducer(RegionalReduceConfig(out_dir=regional_dir, overwrite=force_compute, verbose=verbose, member_quantiles=member_quantiles, landmask_file=land_mask, landmask_var="landfrac"), regions=regions)
exp_dict = build_experiments(str(INPUT_DATA_ROOT))
for group in groups:
    meta=EXPERIMENT_GROUPS[group]; input_dir=base_dir/group
    for run in run_by_group.get(group, (meta["run"],)):
        for obs, variables in selected.items():
            for exp in meta["models"]:
                for var in variables:
                    result = reducer.reduce_one_exp_var(input_dir, group=group, freq=meta["freq"], run=run, obs=obs, period=meta["period"], exp=exp, var=var, nens=int(exp_dict[exp]["nens"]))
                    print("Saved/reused:", result)

